In [ ]:
import os
import re
import sys
import mne
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from mne.preprocessing import ICA
import asrpy
import warnings
warnings.filterwarnings("ignore")  # 忽略所有警告

In [3]:
class CAEEG_preprocessing:
    """
    USTB-MCI EEG 数据预处理类
    按照优化后的EEG预处理流程组织：
    
    A. 质控与设置
        1. 加载数据（保留原始参考信息）
        2. 通道设置：选择 EEG/EOG/ECG，设置单位、坏段标记规则
        3. 设置 montage（电极位置非常重要：后续插值要用）
        4. 临时重参考用于目视检查（不保存、不进入正式分支）
        5. 人工标记坏道 bads（只标记，不插值）
        6. 查找睁/闭眼区段
    
    B. 信号整形（建议先做）
        6. 滤波：高通 0.5/1 Hz，低通 45 Hz，notch 50/60 Hz
        7. 降采样：到目标采样率（低通后做，避免混叠）
    
    C. 伪迹去除（连续数据上做更稳）
        9. 裁剪睁/闭眼段
        10. ASR处理
        11. ICA处理
    
    D. 修补与定稿（强烈建议）
        12. 坏道插值（在 ASR/ICA 之后）
        13. 最终重参考（平均参考 CAR）
        14. 分段和保存
    """

    def __init__(self, l_freq, h_freq, sfreq, keep_channels, save_dir='CAEEG'):
        self.save_dir = save_dir
        self.data_file_path = None
        self.subject = None
        
        # 原始数据
        self.raw = None
        self.events = None
        self.event_dict = None
        
        # 处理参数
        self.l_freq = l_freq  # 高通滤波频率
        self.h_freq = h_freq   # 低通滤波频率
        self.sfreq = sfreq   # 目标采样率
        self.notch_freq = 50  # 陷波滤波频率
        
        # 通道配置
        self.keep_channels = keep_channels
        self.aux_channels = ['ECG', 'HEOR', 'HEOL']  # 辅助通道
        self.eeg_channels = None  # EEG通道列表
        self.bad_channels = []    # 坏导联列表
        
        # 分段信息
        self.open_eyes_tmin = None
        self.open_eyes_tmax = None
        self.closed_eyes_tmin = None
        self.closed_eyes_tmax = None
        self.raw_open = None
        self.raw_closed = None
        self.raw_open_clean = None
        self.raw_closed_clean = None
        
        # 日志文件
        self.log_file = None  # 日志文件对象
        self.log_file_path = None  # 日志文件路径
        self._original_stdout = None  # 原始stdout，用于恢复


    # ========== 工具函数 ==========
    @staticmethod
    def _first_matching(ch_name, pool):
        """大小写无关 + 允许以目标名为前缀的匹配（兼容 -AVG/-REF 后缀）"""
        low = ch_name.lower()
        for c in pool:
            if c.lower() == low:
                return c
        for c in pool:
            if c.lower().startswith(low):
                return c
        return None
    
    def _log_print(self, *args, **kwargs):
        """
        自定义打印函数，同时输出到控制台和日志文件
        """
        # 输出到控制台
        print(*args, **kwargs)
        
        # 输出到日志文件
        if self.log_file is not None:
            # 将参数转换为字符串
            message = ' '.join(str(arg) for arg in args)
            # 写入文件（添加换行符）
            self.log_file.write(message + '\n')
            self.log_file.flush()  # 立即刷新缓冲区，确保内容写入文件
    
    def _init_log_file(self, subject=None, log_file_path=None):
        """
        初始化日志文件
        如果提供了log_file_path，使用该路径（追加模式，用于批量处理）
        否则在save_dir下创建日志文件，文件名为：{subject}_preprocessing_log.txt
        
        参数:
        ----------
        subject : str, optional
            被试名称（如果log_file_path为None时使用）
        log_file_path : str or Path, optional
            日志文件路径。如果提供，将使用该路径（追加模式）
        """
        try:
            # 如果已经初始化过日志文件，直接返回
            if self.log_file is not None:
                return
            
            if log_file_path is not None:
                # 使用提供的日志文件路径（追加模式）
                self.log_file_path = Path(log_file_path)
                # 确保目录存在
                os.makedirs(self.log_file_path.parent, exist_ok=True)
                # 追加模式
                mode = 'a' if self.log_file_path.exists() else 'w'
                self.log_file = open(self.log_file_path, mode, encoding='utf-8')
                
                # 如果是新文件，写入头部信息
                if mode == 'w':
                    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    self.log_file.write("="*80 + "\n")
                    self.log_file.write(f"CAEEG EEG 数据批量预处理日志\n")
                    self.log_file.write(f"开始时间: {timestamp}\n")
                    self.log_file.write("="*80 + "\n\n")
                    self.log_file.flush()
            else:
                # 使用默认路径（基于subject）
                if subject is None:
                    raise ValueError("如果log_file_path为None，必须提供subject参数")
                # 确保save_dir存在
                os.makedirs(self.save_dir, exist_ok=True)
                # 创建日志文件路径
                self.log_file_path = Path(self.save_dir) / f"{subject}_preprocessing_log.txt"
                # 打开日志文件
                self.log_file = open(self.log_file_path, 'w', encoding='utf-8')
                
                # 写入日志文件头部信息（仅在新文件时）
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                self.log_file.write("="*80 + "\n")
                self.log_file.write(f"CAEEG EEG 数据预处理日志\n")
                self.log_file.write(f"被试: {subject}\n")
                self.log_file.write(f"开始时间: {timestamp}\n")
                self.log_file.write("="*80 + "\n\n")
                self.log_file.flush()
            
            # 保存原始的stdout（如果还没有保存）
            if not hasattr(self, '_original_stdout') or self._original_stdout is None:
                self._original_stdout = sys.stdout
            
            # 创建自定义的stdout类，同时输出到控制台和文件
            class TeeOutput:
                def __init__(self, file, original_stdout):
                    self.file = file
                    self.original_stdout = original_stdout
                
                def write(self, text):
                    # 写入控制台
                    self.original_stdout.write(text)
                    # 写入文件
                    if self.file is not None:
                        self.file.write(text)
                        self.file.flush()
                
                def flush(self):
                    self.original_stdout.flush()
                    if self.file is not None:
                        self.file.flush()
            
            # 替换stdout
            sys.stdout = TeeOutput(self.log_file, self._original_stdout)
            
            if log_file_path is None:
                print(f"日志文件已创建: {self.log_file_path}")
        except Exception as e:
            # 如果创建日志文件失败，只输出警告，不影响主流程
            print(f"⚠ 警告: 无法创建日志文件: {e}")
            self.log_file = None
    
    def _close_log_file(self):
        """
        关闭日志文件并恢复原始stdout
        """
        # 恢复原始的stdout
        if hasattr(self, '_original_stdout'):
            sys.stdout = self._original_stdout
        
        if self.log_file is not None:
            try:
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                # 直接写入文件（因为stdout已经恢复）
                self.log_file.write("\n" + "="*80 + "\n")
                self.log_file.write(f"结束时间: {timestamp}\n")
                self.log_file.write("="*80 + "\n")
                self.log_file.close()
                self.log_file = None
                print(f"✓ 日志文件已保存: {self.log_file_path}")
            except Exception as e:
                print(f"⚠ 警告: 关闭日志文件时出错: {e}")

    # ========== A.1: 数据加载（保留原始参考信息）==========
    def load_data_and_check(self):
        """
        A.1: 数据加载与初步检查
        - 加载原始数据文件（BDF文件已包含事件信息）
        - 保留原始参考信息（不进行重参考）
        - 检查采样率、通道数、数据时长
        - 从BDF文件中读取事件标记
        """
        print("\n" + "="*60)
        print("A.1: 数据加载（保留原始参考信息）")
        print("="*60)
        
        # 加载BDF文件（BDF文件通常已包含annotations）
        print(f"正在加载数据文件: {self.data_file_path}")
        self.raw = mne.io.read_raw_bdf(self.data_file_path, preload=True, verbose='error')
        print("✓ 数据文件加载完成")
        
        # 检查基本信息
        print(f"  - 采样率: {self.raw.info['sfreq']} Hz")
        print(f"  - 通道数量: {len(self.raw.ch_names)}")
        print(f"  - 数据时长: {self.raw.times[-1]:.2f} 秒")
        print(f"  - 数据形状: {self.raw.get_data().shape}")
        
        # 检查原始参考信息
        if self.raw.info.get('custom_ref_applied', None) == 0:
            ref_info = "未应用自定义参考"
        else:
            ref_info = f"参考: {self.raw.info.get('custom_ref_applied', '未知')}"
        print(f"  - 原始参考信息: {ref_info}")
        print("  ✓ 保留原始参考信息，暂不进行重参考")
        
        # 从BDF文件中读取事件标记（如果文件已包含annotations）
        annotations = self.raw.annotations
        if len(annotations) > 0:
            print(f"✓ 从BDF文件中读取到 {len(annotations)} 个注解")
        else:
            print("⚠ 警告: BDF文件中未找到注解信息")
        
        # 提取事件
        self.events, self.event_dict = mne.events_from_annotations(
            self.raw, verbose=False
        )
        print(f"  - 事件数量: {len(self.events)}")
        print(f"  - 事件类型: {list(self.event_dict.keys())}")
        
        print("✓ A.1完成: 数据加载（保留原始参考信息）")


    # ========== A.2: 通道设置 ==========
    def select_and_setup_channels(self):
        """
        A.2: 通道设置
        - 选择需要的EEG通道
        - 设置通道类型（EEG、EOG、ECG等）
        - 设置单位、坏段标记规则
        """
        print("\n" + "="*60)
        print("A.2: 通道设置")
        print("="*60)
        
        print(f"原始通道列表 ({len(self.raw.ch_names)} 个):")
        print(f"  {self.raw.ch_names[:10]}..." if len(self.raw.ch_names) > 10 else f"  {self.raw.ch_names}")

        # 1. 筛选通道
        print("\n1. 筛选通道...")
        existing = self.raw.ch_names
        keep_matched = []
        missing = []
        for k in self.keep_channels:
            m = self._first_matching(k, existing)
            if m is None:
                missing.append(k)
            else:
                keep_matched.append(m)
        if missing:
            print(f"  警告: 以下通道未找到，将跳过: {missing}")
        self.raw.pick_channels(keep_matched, ordered=False)
        print(f"  ✓ 保留 {len(keep_matched)} 个通道")
        print(f"  通道列表: {self.raw.ch_names}")

        # 2. 设置通道类型
        print("\n2. 设置通道类型...")
        self.eeg_channels = [ch for ch in self.raw.ch_names if ch not in self.aux_channels]
        print(f"  - EEG通道: {len(self.eeg_channels)} 个， {self.eeg_channels}")
        print(f"  - 辅助通道: {len(self.aux_channels)} 个， {self.aux_channels}")
        
        type_map = {ch: 'eeg' for ch in self.eeg_channels}
        for aux, t in [('ECG', 'ecg'), ('HEOR', 'eog'), ('HEOL', 'eog')]:
            m = self._first_matching(aux, self.raw.ch_names)
            if m is not None:
                type_map[m] = t
        self.raw.set_channel_types(type_map)
        print("  ✓ 通道类型设置完成 (ECG: ecg, HEOR/HEOL: eog, 其余: eeg)")
        
        # 设置单位（确保单位正确）
        print("\n3. 设置通道单位...")
        # MNE会自动处理单位，这里主要是确认
        print("  ✓ 通道单位已设置（MNE自动处理）")
        
        print("\n✓ A.2完成: 通道设置")


    # ========== A.3: 设置 montage ==========
    def setup_montage(self):
        """
        A.3: 设置 montage（电极位置）
        - 电极位置非常重要：后续插值要用
        - 优先加载自定义电极位置信息，否则使用标准模板
        """
        print("\n" + "="*60)
        print("A.3: 设置 montage（电极位置）")
        print("="*60)
        
        try:
            markers = np.array(self.eeg_channels)
            n_ch = len(markers)
            if n_ch == 19 and os.path.exists('1020-19.xlsx'):
                data1020 = pd.read_excel('1020-19.xlsx', index_col=0)
                print("载入 1020-19.xlsx 自定义电极坐标")
            elif n_ch == 20 and os.path.exists('1020-20.xlsx'):
                data1020 = pd.read_excel('1020-20.xlsx', index_col=0)
                print("载入 1020-20.xlsx 自定义电极坐标")
            else:
                data1020 = None
                print(f"未找到匹配通道数 ({n_ch}) 的自定义坐标文件，改用标准模板。")
            if data1020 is not None:
                channels1020 = np.array(data1020.index)
                value1020 = np.array(data1020)
                list_dic = dict(zip(channels1020, value1020))
                montage_1020 = mne.channels.make_dig_montage(
                    ch_pos=list_dic,
                    nasion=[5.27205792e-18, 8.60992398e-02, -4.01487349e-02],
                    lpa=[-0.08609924, 0., -0.04014873],
                    rpa=[0.08609924, 0., -0.04014873]
                )
                # 将蒙太奇直接应用到 raw
                self.raw.set_montage(montage_1020, match_case=False)
                print(f"已加载自定义电极蒙太奇，共 {len(channels1020)} 通道。")
            else:
                # 若无自定义文件则使用标准 10-20 模板
                montage_std = mne.channels.make_standard_montage('standard_1020')
                self.raw.set_montage(montage_std, match_case=False)
                print("已应用 standard_1020 模板蒙太奇。")
        except Exception as e:
            print(f"自定义蒙太奇加载失败: {e}\n 改用标准 10-20 模板。")
            montage_std = mne.channels.make_standard_montage('standard_1020')
            self.raw.set_montage(montage_std, match_case=False)
            print("已应用 standard_1020 模板蒙太奇。")
        
        print("✓ A.3完成: 设置 montage")


    # ========== A.4: 临时重参考用于目视检查（可选）==========
    def apply_temp_rereference(self, ref_type='average', show_plot=False):
        """
        A.4: 临时重参考用于目视检查
        - 不保存、不进入正式分支
        - 仅用于人工检查数据质量
        
        参数:
        ----------
        ref_type : str, default='average'
            参考类型：'average'（平均参考）或其他MNE支持的参考类型
        show_plot : bool, default=False
            是否显示临时重参考后的数据用于检查
        """
        print("\n" + "="*60)
        print("A.4: 临时重参考用于目视检查（可选）")
        print("="*60)
        
        if not show_plot:
            print("跳过临时重参考（show_plot=False）")
            print("✓ A.4完成: 跳过临时重参考")
            return
        
        print(f"应用临时{ref_type}参考用于目视检查...")
        raw_temp = self.raw.copy()
        if ref_type == 'average':
            raw_temp.set_eeg_reference('average', projection=True, ch_type='eeg')
            raw_temp.apply_proj()
        else:
            raw_temp.set_eeg_reference(ref_type)
        print(f"✓ 临时{ref_type}参考完成（仅用于检查，不保存）")
        
        # 显示临时重参考后的数据
        self._show_data_plot(raw_temp, title="临时重参考后的数据（仅用于检查）", show_plot=show_plot)
        
        print("✓ A.4完成: 临时重参考用于目视检查")


    # ========== A.5: 人工标记坏道（只标记，不插值）==========
    def mark_bad_channels(self, bad_channels=None, auto_detect=True, 
                         flat_threshold=1e-15, noisy_threshold=3.0):
        """
        A.5: 人工标记坏道
        - 只标记坏道，不进行插值
        - 插值将在步骤9（ASR/ICA之后）进行
        
        参数:
        ----------
        bad_channels : list of str, optional
            手动指定的坏导联名称列表
        auto_detect : bool, default=True
            是否自动检测坏导联
        flat_threshold : float, default=1e-15
            平坦通道检测阈值（方差阈值）
        noisy_threshold : float, default=3.0
            噪声通道检测阈值（标准差倍数）
        
        返回:
        ----------
        list of str
            被标记的坏导联名称列表
        """
        import warnings
        
        print("\n" + "="*60)
        print("A.5: 人工标记坏道（只标记，不插值）")
        print("="*60)
        
        # 确保只对 EEG 通道进行检测
        eeg_ch_names = [ch for ch in self.raw.ch_names if ch not in self.aux_channels]
        
        # 初始化坏导联列表
        detected_bads = []
        
        # 自动检测坏导联
        if auto_detect:
            print("\n1. 自动检测坏导联...")
            
            # 获取 EEG 数据（抑制 MNE 的警告）
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                data = self.raw.get_data(picks='eeg')
            ch_names = [ch for ch in self.raw.ch_names if ch not in self.aux_channels]
            
            # 1.1 检测平坦通道（方差极小或接近零）
            ch_vars = np.var(data, axis=1)
            flat_chs = [ch_names[i] for i in range(len(ch_names)) 
                       if ch_vars[i] < flat_threshold]
            if flat_chs:
                detected_bads.extend(flat_chs)
                print(f"  ✓ 检测到平坦通道: {flat_chs}")
            
            # 1.2 检测噪声通道（使用标准差方法，基于中位数绝对偏差）
            ch_stds = np.std(data, axis=1)
            median_std = np.median(ch_stds)
            mad_std = np.median(np.abs(ch_stds - median_std))  # 中位数绝对偏差
            
            # 使用阈值检测异常通道（过高或过低的噪声）
            upper_threshold = median_std + noisy_threshold * mad_std
            lower_threshold = max(0, median_std - noisy_threshold * mad_std)
            
            noisy_chs = [ch_names[i] for i in range(len(ch_names)) 
                        if ch_stds[i] > upper_threshold or ch_stds[i] < lower_threshold]
            
            if noisy_chs:
                detected_bads.extend(noisy_chs)
                print(f"  ✓ 检测到噪声通道: {noisy_chs}")
            
            # 1.3 检测异常幅值通道（超出合理范围）
            ch_max = np.max(np.abs(data), axis=1)
            median_max = np.median(ch_max)
            mad_max = np.median(np.abs(ch_max - median_max))
            max_threshold = median_max + noisy_threshold * mad_max
            
            extreme_chs = [ch_names[i] for i in range(len(ch_names)) 
                          if ch_max[i] > max_threshold]
            
            if extreme_chs:
                detected_bads.extend(extreme_chs)
                print(f"  ✓ 检测到异常幅值通道: {extreme_chs}")
        
        # 合并手动指定和自动检测的坏导联
        if bad_channels is not None:
            print("\n2. 处理手动指定的坏导联...")
            # 验证手动指定的通道是否存在
            valid_bads = [ch for ch in bad_channels if ch in eeg_ch_names]
            if valid_bads:
                detected_bads.extend(valid_bads)
                print(f"  ✓ 手动指定的坏导联: {valid_bads}")
            invalid_bads = [ch for ch in bad_channels if ch not in eeg_ch_names]
            if invalid_bads:
                print(f"  警告: 以下指定的坏导联不存在或不是 EEG 通道，将被忽略: {invalid_bads}")
        
        # 去重
        detected_bads = list(set(detected_bads))
        
        # 确保坏导联都是 EEG 通道
        detected_bads = [ch for ch in detected_bads if ch in eeg_ch_names]
        
        if not detected_bads:
            print("\n✓ 未检测到坏导联")
            print("✓ A.5完成: 人工标记坏道（只标记，不插值）")
            self.bad_channels = []
            return []
        
        print(f"\n3. 标记坏导联（不插值）...")
        print(f"  检测到 {len(detected_bads)} 个坏导联: {detected_bads}")
        
        # 只标记坏导联，不进行插值
        self.raw.info['bads'] = detected_bads
        self.bad_channels = detected_bads
        print(f"  ✓ 已标记 {len(detected_bads)} 个坏导联（将在步骤9进行插值）")
        
        print("\n✓ A.5完成: 人工标记坏道（只标记，不插值）")
        return detected_bads


    # ========== D.12: 坏道插值（在ASR/ICA之后）==========
    def interpolate_bad_channels(self):
        """
        D.12: 坏道插值
        - 在 ASR/ICA 之后进行插值
        - 使用之前标记的坏道进行插值
        """
        import warnings
        from contextlib import redirect_stdout, redirect_stderr
        from io import StringIO
        
        print("\n" + "="*60)
        print("D.12: 坏道插值（在ASR/ICA之后）")
        print("="*60)
        
        # 检查是否有标记的坏道
        if not self.bad_channels or len(self.bad_channels) == 0:
            print("未检测到坏导联，跳过插值")
            print("✓ D.12完成: 坏道插值（跳过）")
            return
        
        print(f"插值 {len(self.bad_channels)} 个坏导联: {self.bad_channels}")
        
        # 确保坏道标记存在
        self.raw.info['bads'] = self.bad_channels
        
        # 执行插值（需要有效的电极位置信息）
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                f = StringIO()
                with redirect_stdout(f), redirect_stderr(f):
                    self.raw.interpolate_bads(reset_bads=True, mode='accurate', verbose=False)
            print(f"  ✓ 成功插值 {len(self.bad_channels)} 个坏导联")
            # 插值后清除坏导联标记
            self.raw.info['bads'] = []
            self.bad_channels = []
        except Exception as e:
            print(f"  ✗ 插值失败: {e}")
            print("  提示: 插值需要有效的电极位置信息（montage）")
            raise
        
        print("✓ D.12完成: 坏道插值")


    def define_segments(self):
        """
        A.6: 查找睁/闭眼区段
        - 从事件中查找睁/闭眼区段的时间范围
        - 注意：此函数在降采样之前调用，使用原始采样率的时间点
        """
        print("\n" + "="*60)
        print("A.6: 查找睁/闭眼区段")
        print("="*60)
        
        if self.events is None or len(self.events) == 0:
            raise ValueError("未发现事件，无法定义分段。")

        def _get_onsets(label):
            """获取事件的时间戳（秒）"""
            if label not in self.event_dict:
                return []
            code = self.event_dict[label]
            # 获取事件索引
            event_indices = self.events[self.events[:, 2] == code][:, 0]
            # 转换为时间（秒）
            sfreq = self.raw.info['sfreq']
            return [idx / sfreq for idx in event_indices]

        # 根据事件字典自动选择使用哪组事件标签
        event_labels = list(self.event_dict.keys())
        # 检查事件字典中是否包含 '11', '12', '13', '14'
        if all(label in event_labels for label in ['11', '12', '13', '14']):
            # 使用 '11', '12', '13', '14' 这组事件标签
            print("检测到事件标签: '11', '12', '13', '14'")
            open_start_times = _get_onsets('11')
            open_end_times = _get_onsets('12')
            closed_start_times = _get_onsets('13')
            closed_end_times = _get_onsets('14')
        else:
            # 否则使用 '1', '2', '16', '17' 这组事件标签
            print("检测到事件标签: '1', '2', '16', '17'")
            open_start_times = _get_onsets('1')
            open_end_times = _get_onsets('2')
            closed_start_times = _get_onsets('16')
            closed_end_times = _get_onsets('17')

        if not (len(open_start_times) and len(open_end_times) and 
                len(closed_start_times) and len(closed_end_times)):
            raise ValueError(f"事件不完整：open_start={len(open_start_times)}, open_end={len(open_end_times)}, "
                             f"closed_start={len(closed_start_times)}, closed_end={len(closed_end_times)}")

        # 取每类的第一段（如有多段可自行扩展逻辑）
        self.open_eyes_tmin = open_start_times[0]
        self.open_eyes_tmax = open_end_times[0]
        self.closed_eyes_tmin = closed_start_times[0]
        self.closed_eyes_tmax = closed_end_times[0]

        print(f"\n睁眼段: {self.open_eyes_tmin:.3f}s → {self.open_eyes_tmax:.3f}s "
              f"(时长={self.open_eyes_tmax - self.open_eyes_tmin:.3f}s)")
        print(f"闭眼段: {self.closed_eyes_tmin:.3f}s → {self.closed_eyes_tmax:.3f}s "
              f"(时长={self.closed_eyes_tmax - self.closed_eyes_tmin:.3f}s)")
        
        print("\n✓ A.6完成: 查找睁/闭眼区段")


    # ========== B.6: 滤波 ==========
    def apply_filtering(self, show_plot=False):
        """
        B.6: 滤波（Filtering）
        - 高通滤波：0.5 Hz（或 1 Hz，如果漂移重/要做 ICA 更稳）
        - 低通滤波：45 Hz
        - 陷波滤波：去除工频干扰（50/60 Hz及其谐波）
        
        参数:
        ----------
        show_plot : bool, default=False
            是否在最后显示处理后的数据。如果为True，将使用MNE的plot方法显示数据。
        """
        print("\n" + "="*60)
        print("B.6: 滤波")
        print("="*60)
        
        # 1. 陷波滤波：去除工频干扰
        print("\n1. 陷波滤波（去除工频干扰）...")
        sfreq = self.raw.info['sfreq']
        nyq = sfreq / 2.0
        if nyq > self.notch_freq:
            base = float(self.notch_freq)
            freqs = np.arange(base, nyq, base)  # 严格小于 Nyquist
            print(f"  去除频率: {freqs.tolist()} Hz")
            # IIR 不支持多阻带一次完成——逐个频点循环
            for f0 in freqs:
                self.raw.notch_filter(
                    freqs=f0,
                    picks='all',
                    method='iir',
                    iir_params={'order': 4, 'ftype': 'butter'},
                    verbose=False
                )
            print(f"  ✓ 陷波滤波完成 (IIR, 逐点): {freqs.tolist()} Hz")
        else:
            print(f"  Nyquist频率 ({nyq:.1f} Hz) < {self.notch_freq} Hz，跳过陷波滤波")
        
        # 2. 带通滤波
        print(f"\n2. 带通滤波 ({self.l_freq}-{self.h_freq} Hz)...")
        self.raw.filter(
            l_freq=self.l_freq, 
            h_freq=self.h_freq, 
            picks='all',
            fir_design='firwin', 
            phase='zero', 
            fir_window='hamming',
            verbose=False
        )
        print(f"  ✓ 带通滤波完成: {self.l_freq}-{self.h_freq} Hz")
        
        print("\n✓ B.6完成: 滤波")
        
        # 如果show_plot为True，显示处理后的数据
        self._show_data_plot(self.raw, title="滤波后的数据", show_plot=show_plot)

    # ========== B.7: 降采样 ==========
    def apply_downsampling(self, show_plot=False):
        """
        B.7: 降采样（Downsampling）
        - 降低采样率以减少数据量和计算量
        - 在低通滤波后做，避免混叠
        
        参数:
        ----------
        show_plot : bool, default=False
            是否在最后显示处理后的数据。如果为True，将使用MNE的plot方法显示数据。
        """
        print("\n" + "="*60)
        print("B.7: 降采样")
        print("="*60)
        
        current_sfreq = self.raw.info['sfreq']
        print(f"当前采样率: {current_sfreq} Hz")
        print(f"目标采样率: {self.sfreq} Hz")
        
        if abs(current_sfreq - self.sfreq) > 1e-6:
            print(f"正在降采样到 {self.sfreq} Hz...")
            self.raw.resample(sfreq=self.sfreq, npad='auto', verbose=False)
            print(f"✓ 降采样完成: {current_sfreq} Hz → {self.sfreq} Hz")
        else:
            print(f"✓ 已是目标采样率 {self.sfreq} Hz，跳过降采样")
        
        print(f"处理后数据形状: {self.raw.get_data().shape}")
        print("\n✓ B.7完成: 降采样")
        
        # 如果show_plot为True，显示处理后的数据
        self._show_data_plot(self.raw, title="降采样后的数据", show_plot=show_plot)


    def crop_segments(self):
        """
        C.9: 裁剪睁/闭眼段
        - 从处理后的 raw 中裁剪睁/闭眼段
        - 用于后续的ASR和ICA处理
        """
        print("\n" + "="*60)
        print("C.9: 裁剪睁/闭眼段")
        print("="*60)
        
        self.raw_open = self.raw.copy().crop(tmin=self.open_eyes_tmin, tmax=self.open_eyes_tmax)
        self.raw_closed = self.raw.copy().crop(tmin=self.closed_eyes_tmin, tmax=self.closed_eyes_tmax)
        print(f"截取后睁眼段形状: {self.raw_open.get_data().shape}")
        print(f"截取后闭眼段形状: {self.raw_closed.get_data().shape}")
        
        print("✓ C.9完成: 裁剪睁/闭眼段")


    def find_stable_calibration_segment(self, raw_eeg, search_start=0.0, search_duration=60.0,
                                       target_duration=30.0, window_len=2.0, 
                                       step_len=0.5, amplitude_threshold=200.0,
                                       high_freq_range=(35, 45)):
        """
        在记录的前段中寻找相对干净的数据段用于ASR校准。
        
        使用滑动窗口方法，计算多个稳定性指标，选择最稳定的窗口拼接成校准段。
        
        参数:
        ----------
        raw_eeg : mne.io.Raw
            要搜索的EEG数据（只包含EEG通道，已应用高通滤波）
        search_start : float, default=0.0
            搜索范围起始点（秒），默认从记录开始
        search_duration : float, default=60.0
            搜索范围时长（秒），在记录的前search_duration秒中搜索
            建议范围：30-120秒，默认60秒
        target_duration : float, default=30.0
            目标校准段时长（秒），最终拼接的校准段长度
        window_len : float, default=2.0
            滑动窗口长度（秒），建议范围：2-5秒
        step_len : float, default=0.5
            滑动窗口步长（秒）
        amplitude_threshold : float, default=200.0
            振幅异常阈值（μV），超过此阈值的窗口将被排除
        high_freq_range : tuple, default=(30, 80)
            高频功率检测范围（Hz），用于检测肌电伪迹
        
        返回:
        ----------
        raw_calibration : mne.io.Raw
            找到的稳定校准段，如果未找到足够的数据则返回None
        """
        from scipy import signal as sp_signal
        
        sfreq = raw_eeg.info['sfreq']
        total_duration = raw_eeg.times[-1]
        
        # 限制搜索范围
        search_end = search_duration + search_start
        if search_end > total_duration:
            print(f"⚠ 警告：数据时长({total_duration:.1f}s) < 搜索范围截止端({search_end:.1f}s)，无法进行稳定段搜索")
            return None
        
        print(f"\n搜索稳定校准段：")
        print(f"  搜索范围：{search_start:.1f} 秒 到 {search_end:.1f} 秒")
        print(f"  目标时长：{target_duration:.1f} 秒")
        print(f"  窗口长度：{window_len:.1f} 秒，步长：{step_len:.1f} 秒")
        
        # 裁剪搜索范围
        raw_search = raw_eeg.copy().crop(tmin=search_start, tmax=search_end)
        data = raw_search.get_data()  # (n_channels, n_times)
        n_channels, n_times = data.shape
        
        # 计算窗口参数
        win_samples = int(window_len * sfreq)
        step_samples = int(step_len * sfreq)
        
        # 存储所有窗口的得分和索引
        window_scores = []
        window_indices = []
        
        # 遍历所有窗口
        for start_idx in range(0, n_times - win_samples + 1, step_samples):
            end_idx = start_idx + win_samples
            window_data = data[:, start_idx:end_idx]  # (n_channels, win_samples)
            
            # 初始化得分
            score = 0.0
            is_valid = True
            
            # ========== 指标1: 整体幅度/RMS ==========
            # 计算每个通道的RMS
            ch_rms = np.sqrt(np.mean(window_data ** 2, axis=1))  # (n_channels,)
            # 使用中位数作为整体幅度指标（更稳健）
            median_rms = np.median(ch_rms)
            # RMS越小越好，归一化到0-1范围（相对于所有窗口）
            # 这里先记录原始值，后面统一归一化
            rms_score = median_rms
            
            # ========== 指标2: 高频功率（肌电检测）==========
            # 使用Welch方法估计功率谱
            high_freq_power = 0.0
            for ch_idx in range(n_channels):
                freqs, psd = sp_signal.welch(
                    window_data[ch_idx, :], 
                    fs=sfreq, 
                    nperseg=min(win_samples // 4, 256),
                    noverlap=None
                )
                # 计算高频带平均功率
                freq_mask = (freqs >= high_freq_range[0]) & (freqs <= high_freq_range[1])
                if np.any(freq_mask):
                    high_freq_power += np.mean(psd[freq_mask])
            high_freq_power /= n_channels  # 平均所有通道
            # 高频功率越高，得分越高（惩罚项）
            high_freq_score = high_freq_power
            
            # ========== 指标3: 通道间一致性 ==========
            # 计算各通道RMS的标准差/变异系数
            rms_std = np.std(ch_rms)
            rms_mean = np.mean(ch_rms)
            if rms_mean > 0:
                rms_cv = rms_std / rms_mean  # 变异系数
            else:
                rms_cv = 0.0
            # 变异系数越大，说明通道间差异越大，越不稳定
            consistency_score = rms_cv
            
            # ========== 指标4: 简单异常检测（阈值）==========
            # 检查是否有通道振幅超过阈值
            max_amplitude = np.max(np.abs(window_data))
            if max_amplitude > amplitude_threshold:
                # 直接标记为不可用
                is_valid = False
                continue
            
            # 检查是否有饱和段（接近ADC最大值，通常±32768或类似值）
            # 假设饱和阈值为±30000（可根据实际ADC范围调整）
            saturation_threshold = 30000.0
            if np.max(np.abs(window_data)) > saturation_threshold * 0.9:  # 90%的ADC范围
                is_valid = False
                continue
            
            # ========== 综合得分 ==========
            # 将各项指标归一化并加权组合
            # 注意：这里先记录原始值，后面统一归一化
            # 得分越小越好
            score = {
                'rms': rms_score,
                'high_freq': high_freq_score,
                'consistency': consistency_score,
                'start_idx': start_idx,
                'end_idx': end_idx,
                'start_time': start_idx / sfreq,
                'end_time': end_idx / sfreq
            }
            
            window_scores.append(score)
            window_indices.append((start_idx, end_idx))
        
        if len(window_scores) == 0:
            print(f"⚠ 警告：未找到有效的窗口，无法构建校准段")
            return None
        
        # 归一化各项指标到0-1范围，然后加权组合
        rms_values = [s['rms'] for s in window_scores]
        high_freq_values = [s['high_freq'] for s in window_scores]
        consistency_values = [s['consistency'] for s in window_scores]
        
        # 归一化（使用min-max归一化）
        rms_min, rms_max = np.min(rms_values), np.max(rms_values)
        high_freq_min, high_freq_max = np.min(high_freq_values), np.max(high_freq_values)
        consistency_min, consistency_max = np.min(consistency_values), np.max(consistency_values)
        
        # 计算归一化得分（越小越好）
        for i, s in enumerate(window_scores):
            if rms_max > rms_min:
                norm_rms = (s['rms'] - rms_min) / (rms_max - rms_min)
            else:
                norm_rms = 0.0
            
            if high_freq_max > high_freq_min:
                norm_high_freq = (s['high_freq'] - high_freq_min) / (high_freq_max - high_freq_min)
            else:
                norm_high_freq = 0.0
            
            if consistency_max > consistency_min:
                norm_consistency = (s['consistency'] - consistency_min) / (consistency_max - consistency_min)
            else:
                norm_consistency = 0.0
            
            # 加权组合（权重可调）
            # RMS权重：0.4，高频功率权重：0.3，一致性权重：0.3
            s['final_score'] = 0.4 * norm_rms + 0.3 * norm_high_freq + 0.3 * norm_consistency
        
        # 按得分排序（得分越小越好）
        window_scores.sort(key=lambda x: x['final_score'])
        
        # 选择最干净的窗口，拼接直到达到目标时长
        selected_windows = []
        accumulated_duration = 0.0
        
        for score_info in window_scores:
            win_duration = score_info['end_time'] - score_info['start_time']
            if accumulated_duration + win_duration <= target_duration:
                selected_windows.append(score_info)
                accumulated_duration += win_duration
                if accumulated_duration >= target_duration:
                    break
        
        if len(selected_windows) == 0:
            print(f"⚠ 警告：未找到足够的稳定窗口，无法构建校准段")
            return None
        
        # 按时间顺序排序选中的窗口
        selected_windows.sort(key=lambda x: x['start_time'])
        
        # 拼接选中的窗口
        calibration_segments = []
        for win_info in selected_windows:
            seg = raw_search.copy().crop(
                tmin=win_info['start_time'], 
                tmax=win_info['end_time']
            )
            calibration_segments.append(seg)
        
        # 合并所有段
        if len(calibration_segments) == 1:
            raw_calibration = calibration_segments[0]
        else:
            raw_calibration = mne.concatenate_raws(calibration_segments, preload=True)
        
        actual_duration = raw_calibration.times[-1]
        print(f"✓ 找到稳定校准段：")
        print(f"  选中窗口数：{len(selected_windows)}")
        print(f"  实际时长：{actual_duration:.1f} 秒")
        print(f"  平均得分：{np.mean([w['final_score'] for w in selected_windows]):.3f}")
        
        return raw_calibration


    def _asr_preprocess_segment(self, raw_seg, seg_name, cutoff=20, show_plot=False):
        """
        对单个分段应用ASR（Artifact Subspace Reconstruction）处理
        
        使用asrpy库实现ASR，用于清理EEG信号中的大伪迹。
        ASR要求先进行高通滤波（1 Hz），然后使用fit()校准和transform()应用。
        
        参数:
        ----------
        raw_seg : mne.io.Raw
            要处理的原始数据段
        seg_name : str
            段的名称（用于打印信息）
        cutoff : float, default=20
            ASR截止阈值（标准差倍数），用于检测异常时间段。值越大越保守。
            建议范围：10-30，默认20
        show_plot : bool or str, default=False
            是否显示处理前后的对比图
            - False: 不显示
            - True: 只显示处理后的数据
            - 'comparison': 显示处理前后的对比
        
        返回:
        ----------
        raw_seg_clean : mne.io.Raw
            清理后的数据
        """
        print(f"\n--- ASR 处理 {seg_name} 段 ---")
        
        try:
            # 1. 提取EEG通道（ASR只处理EEG通道）
            raw_eeg = raw_seg.copy().pick(self.eeg_channels)
            print(f"{seg_name} 原始EEG数据形状: {raw_eeg.get_data().shape}")
            
            # 2. ASR要求高通滤波（通常1-2 Hz）
            # 使用1 Hz高通滤波，与ICA预处理保持一致
            print(f"{seg_name} 应用高通滤波（1 Hz，ASR要求）...")
            raw_eeg_filtered = raw_eeg.copy().filter(
                l_freq=1.0, h_freq=None, picks='eeg', method='iir',
                iir_params=dict(ftype='butter', order=5), phase='zero', verbose=False)
            
            # 3. 创建ASR对象并校准
            print(f"{seg_name} 使用asrpy库进行ASR处理（cutoff={cutoff}）...")
            sfreq = raw_eeg_filtered.info['sfreq']
            asr = asrpy.ASR(sfreq=sfreq, cutoff=cutoff)
            
            # 使用find_stable_calibration_segment找到相对干净的数据段进行校准
            print(f"{seg_name} 搜索稳定校准段...")
            raw_calibration = self.find_stable_calibration_segment(
                raw_eeg_filtered, 
                search_duration=60.0,  # 在前60秒中搜索
                target_duration=30.0,   # 目标30秒校准段
                window_len=2.0,         # 2秒窗口
                step_len=0.5,           # 0.5秒步长
                amplitude_threshold=200.0,  # 200 μV阈值
                high_freq_range=(35, 45)    # 35-45 Hz高频范围
            )
            
            # 如果找到稳定段，使用它进行校准；否则使用整段数据
            if raw_calibration is not None:
                print(f"{seg_name} 使用找到的稳定段校准ASR模型...")
                asr.fit(raw_calibration)
            else:
                print(f"{seg_name} ⚠ 未找到稳定段，使用整段数据校准（可能效果较差）...")
                asr.fit(raw_eeg_filtered)
            
            # 4. 应用ASR变换
            print(f"{seg_name} 应用ASR变换...")
            raw_eeg_asr = asr.transform(raw_eeg_filtered)
            
            # 5. 如果需要保留AUX通道，将它们加回去
            aux_present = [ch for ch in self.aux_channels if self._first_matching(ch, raw_seg.ch_names)]
            if aux_present:
                aux_raw = raw_seg.copy().pick(aux_present)
                raw_seg_clean = raw_eeg_asr.add_channels([aux_raw], force_update_info=True)
            else:
                raw_seg_clean = raw_eeg_asr
            
            print(f"{seg_name} ✓ ASR处理完成")
            
            # 如果show_plot为True，显示处理前后的对比
            if show_plot:
                if isinstance(show_plot, str) and show_plot.lower() == 'comparison':
                    self._show_data_comparison(
                        raw_before=raw_seg,
                        raw_after=raw_seg_clean,
                        title_before=f"{seg_name} 段 - ASR处理前",
                        title_after=f"{seg_name} 段 - ASR处理后",
                        show_comparison=True
                    )
                else:
                    self._show_data_plot(raw_seg_clean, title=f"{seg_name} 段 - ASR处理后的数据", show_plot=True)
            
            return raw_seg_clean
            
        except Exception as e:
            print(f"{seg_name} ⚠ ASR处理出错: {e}")
            print(f"  将返回原始数据，跳过ASR处理")
            import traceback
            traceback.print_exc()
            return raw_seg.copy()


    def apply_asr(self, cutoff=20, show_plot=False):
        """
        C.10: ASR伪迹清理
        对睁眼和闭眼段应用ASR处理。
        - ASR适合运动/瞬时高幅伪迹多的情况
        - 在连续数据上做更稳
        
        参数:
        ----------
        cutoff : float, default=20
            ASR截止阈值（标准差倍数），用于检测异常时间段。值越大越保守。
            建议范围：10-30，默认20
        show_plot : bool or str, default=False
            是否显示处理后的数据。
            - False: 不显示
            - True: 只显示处理后的数据
            - 'comparison': 显示处理前后的对比
        """
        print("\n" + "="*60)
        print("C.10: ASR伪迹清理")
        print("="*60)
        
        self.raw_open = self._asr_preprocess_segment(
            self.raw_open, "睁眼", 
            cutoff=cutoff, show_plot=show_plot
        )
        self.raw_closed = self._asr_preprocess_segment(
            self.raw_closed, "闭眼",
            cutoff=cutoff, show_plot=show_plot
        )
        
        print("\n✓ C.10完成: ASR伪迹清理")


    def _ica_preprocess_segment(self, raw_seg, seg_name, show_plot=False, 
                                eog_threshold=3.0, ecg_threshold='auto', 
                                emg_threshold=0.7, max_exclude_ratio=0.2,
                                use_iclabel=True, keep_aux_channels=False):
        """
        在单个分段上做 ICA 去 ECG/EOG/EMG：
        - 仅在 EEG 通道拟合 ICA（对 1 Hz 高通副本）
        - EOG: HEOR/HEOL；ECG: ECG；EMG: find_bads_muscle
        - 使用更保守的策略，避免过度排除组件
        - 清理后的 EEG 与 AUX 重新合并
        
        参数:
        ----------
        raw_seg : mne.io.Raw
            要处理的原始数据段
        seg_name : str
            段的名称（用于打印信息）
        show_plot : bool or str, default=False
            是否在最后显示处理后的数据。
            - False: 不显示
            - True: 只显示处理后的数据
            - 'comparison': 显示处理前后的对比（先显示处理前，再显示处理后）
        eog_threshold : float, default=3.0
            眼动伪迹检测阈值（标准差倍数），越高越保守
        ecg_threshold : float or str, default='auto'
            心跳伪迹检测阈值，'auto'表示自动阈值
        emg_threshold : float, default=0.7
            肌电伪迹检测阈值（0-1之间），越高越保守，避免过度排除。建议0.7-0.8
        max_exclude_ratio : float, default=0.3
            最大排除组件比例（0-1之间），避免一次性排除太多组件。例如0.3表示最多排除30%
        use_iclabel : bool, default=True
            是否使用ICLabel方法（需要安装mne-icalabel）。如果True，使用ICLabel自动标注
        keep_aux_channels : bool, default=False
            是否保留AUX通道（ECG/EOG）。如果为False，只返回EEG通道。
            注意：ICA处理时只对EEG通道进行处理，AUX通道不会被ICA处理。
            如果设置为False，最终返回的数据将只包含EEG通道，不包含AUX通道。
        
        返回:
        ----------
        raw_seg_clean : mne.io.Raw
            清理后的数据
        """
        print(f"\n--- ICA 处理 {seg_name} 段 ---")

        # EEG-only 视图
        raw_eeg = raw_seg.copy().pick_channels(self.eeg_channels)

        # 用 1 Hz 高通副本拟合 ICA（更稳定）
        raw_for_ica = raw_eeg.copy().filter(
            l_freq=1.0, h_freq=None, picks='eeg', method='iir',
            iir_params=dict(ftype='butter', order=5), phase='zero', verbose=False)

        # ICLabel官方建议：在ICA拟合前对数据做平均参考
        # 注意：在步骤4中已经对整个raw应用了平均参考，所以raw_seg应该已经应用了平均参考
        # 但是，为了确保ICLabel的最佳效果和代码的健壮性，我们再次确保数据应用了平均参考
        # set_eeg_reference('average') 是幂等的，如果已经应用了平均参考，再次应用不会改变数据
        # 这样可以确保即使步骤4的参考方式不同，ICLabel也能正常工作
        if use_iclabel:
            # 由于set_eeg_reference('average')是幂等的，直接应用即可
            # 如果数据已经应用了平均参考，再次应用不会改变数据
            # 这样可以确保ICLabel的要求得到满足
            raw_for_ica.set_eeg_reference('average', projection=False, verbose=False)
            print(f"{seg_name} ✓ 已确保数据应用了平均参考（ICLabel要求）")

        # 分量数：解释 99% 方差，或也可以给一个固定整数
        n_components = 0.999 # min(20, len(self.eeg_channels))  # 不超过通道数
        
        # ICLabel官方建议：使用 extended-infomax ICA
        # 如果使用ICLabel，使用picard方法（extended=True）
        # 如果不使用ICLabel，可以使用fastica（更快）
        if use_iclabel:
            # ICLabel推荐：使用 extended-infomax ICA
            # method='picard' with extended=True 或 method='infomax' with extended=True
            from picard import picard
            print(f"{seg_name} 使用 extended-infomax ICA（ICLabel推荐）...")
            ica = ICA(
                n_components=n_components, 
                method='picard',  # 或使用 'infomax'
                fit_params=dict(ortho=False, extended=True),  # extended=True 是ICLabel的关键要求
                random_state=97, 
                max_iter='auto', 
                verbose=False
            )
        else:
            # 不使用ICLabel时，可以使用fastica（更快）
            print(f"{seg_name} 使用 fastica 方法...")
            ica = ICA(
                n_components=n_components, 
                method='fastica', 
                random_state=97, 
                max_iter='auto', 
                verbose=False
            )
        
        ica.fit(raw_for_ica, picks='eeg', reject_by_annotation=True, decim=3, verbose=False)
        n_ics = len(ica.get_components().T)
        print(f"{seg_name} ICA 拟合完成，共 {n_ics} 个独立成分。")

        # 初始化变量
        eog_inds, ecg_inds, emg_inds = [], [], []
        eog_scores, ecg_scores, emg_scores = None, None, None
        
        # 如果使用ICLabel方法
        if use_iclabel:
            try:
                from mne_icalabel import label_components
                ic_labels = label_components(raw_for_ica, ica, method='iclabel')
                labels = ic_labels['labels']
                probs = ic_labels['y_pred_proba']
                # 格式化输出：每个伪迹组件后面跟着它的概率
                print(f"{seg_name} ICLabel检测结果:")
                for i, (label, prob) in enumerate(zip(labels, probs)):
                    print(f"  IC{i}: {label} (概率={prob:.3f})")
                
                # ICLabel阈值（更保守）
                artifact_thresholds = {
                    'eye blink': 0.6,         # 眼电：> 0.7 ~ 0.9
                    'muscle artifact': 0.6,   # 肌电：> 0.6 ~ 0.8
                    'heart beat': 0.7,        # 心电：> 0.7 ~ 0.9
                    'line noise': 0.7,        # 工频噪声：> 0.7
                    'channel noise': 0.7      # 通道噪声：> 0.7
                }
                
                to_exclude = []
                excluded_info = []
                for i, (label, prob) in enumerate(zip(labels, probs)):
                    if label in artifact_thresholds and prob >= artifact_thresholds[label]:
                        to_exclude.append(i)
                        excluded_info.append(f"IC{i}: {label} (概率={prob:.3f})")
                        # 分类到对应的列表中
                        if label == 'eye blink':
                            eog_inds.append(i)
                        elif label == 'heart beat':
                            ecg_inds.append(i)
                        elif label == 'muscle artifact':
                            emg_inds.append(i)
                
                if excluded_info:
                    print(f"{seg_name} ICLabel检测到的伪迹组件:")
                    for info in excluded_info:
                        print(f"  - {info}")
                    exclude = sorted(to_exclude)
                else:
                    print(f"{seg_name} ICLabel未检测到需要排除的伪迹组件")
                    exclude = []
            except ImportError:
                print(f"⚠ 警告：未安装mne-icalabel，使用传统方法")
                use_iclabel = False
        
        # 使用传统方法
        if not use_iclabel:
            # -------- EOG: HEOR/HEOL --------
            eog_inds, eog_scores = ica.find_bads_eog(raw_seg, ch_name=['HEOR', 'HEOL'], 
                                                      threshold=eog_threshold, verbose=False)
            if eog_inds:
                print(f"{seg_name} 眼动伪迹组件: {eog_inds}")
                if eog_scores is not None and len(eog_scores) > 0:
                    max_score_idx = np.argmax([np.max(s) if len(s) > 0 else 0 for s in eog_scores])
                    if max_score_idx < len(eog_scores) and len(eog_scores[max_score_idx]) > 0:
                        print(f"  最高得分: {np.max(eog_scores[max_score_idx]):.3f}")
            else:
                print(f"{seg_name} 未检测到眼动伪迹组件")

            # -------- ECG --------
            ecg_inds, ecg_scores = ica.find_bads_ecg(raw_seg, ch_name='ECG', 
                                                      threshold=ecg_threshold, l_freq=8, h_freq=16, verbose=False)
            if ecg_inds:
                print(f"{seg_name} 心跳伪迹组件: {ecg_inds}")
                if ecg_scores is not None and len(ecg_scores) > 0:
                    max_score = np.max(ecg_scores)
                    print(f"  最高得分: {max_score:.3f}")
            else:
                print(f"{seg_name} 未检测到心跳伪迹组件")

            # -------- EMG（肌电）--------
            # 使用更保守的阈值（从0.5提高到0.7），避免过度排除
            emg_inds, emg_scores = ica.find_bads_muscle(raw_eeg, threshold=emg_threshold, verbose=False)
            if emg_inds:
                print(f"{seg_name} 肌电伪迹组件: {emg_inds} (阈值={emg_threshold:.2f})")
                if emg_scores is not None and len(emg_scores) > 0:
                    # 显示每个被标记组件的得分
                    for idx in emg_inds:
                        if idx < len(emg_scores):
                            print(f"  IC{idx} 肌电得分: {emg_scores[idx]:.3f}")
            else:
                print(f"{seg_name} 未检测到肌电伪迹组件 (阈值={emg_threshold:.2f})")

            # 合并排除
            exclude = sorted(set(eog_inds + ecg_inds + emg_inds))

        # 限制排除的组件数量，避免过度排除
        max_exclude = int(n_ics * max_exclude_ratio)
        if len(exclude) > max_exclude:
            print(f"⚠ 警告：检测到 {len(exclude)} 个伪迹组件，超过最大排除比例 {max_exclude_ratio:.0%} ({max_exclude}个)")
            print(f"  将只排除得分最高的 {max_exclude} 个组件")
            
            # 计算每个组件的综合得分（优先级：EOG > ECG > EMG）
            component_scores = {}
            for idx in exclude:
                score = 0
                # EOG优先级最高
                if idx in eog_inds and eog_scores is not None:
                    if isinstance(eog_scores, list) and len(eog_scores) > 0:
                        # eog_scores是一个列表的列表，需要找到对应索引的最大值
                        for score_list in eog_scores:
                            if isinstance(score_list, (list, np.ndarray)) and len(score_list) > idx:
                                score += 100 + (score_list[idx] if idx < len(score_list) else 0) * 10
                    elif isinstance(eog_scores, np.ndarray) and idx < len(eog_scores):
                        score += 100 + eog_scores[idx] * 10
                    else:
                        score += 100  # 如果无法获取得分，给默认高分
                # ECG次之
                if idx in ecg_inds and ecg_scores is not None:
                    if isinstance(ecg_scores, (list, np.ndarray)) and idx < len(ecg_scores):
                        score += 50 + ecg_scores[idx] * 10
                    else:
                        score += 50
                # EMG优先级最低
                if idx in emg_inds and emg_scores is not None:
                    if isinstance(emg_scores, (list, np.ndarray)) and idx < len(emg_scores):
                        score += emg_scores[idx] * 10
                    else:
                        score += 5
                
                # 如果组件在所有类别中都没有得分，给一个低分
                if score == 0:
                    score = 1
                component_scores[idx] = score
            
            # 按得分排序，只保留得分最高的max_exclude个
            sorted_exclude = sorted(component_scores.items(), key=lambda x: x[1], reverse=True)
            exclude = [idx for idx, score in sorted_exclude[:max_exclude]]
            print(f"  最终排除的组件: {exclude}")
            print(f"  排除原因: EOG={[i for i in exclude if i in eog_inds]}, "
                  f"ECG={[i for i in exclude if i in ecg_inds]}, "
                  f"EMG={[i for i in exclude if i in emg_inds]}")

        if exclude:
            print(f"{seg_name} 将排除 ICA 成分 ({len(exclude)}/{n_ics}): {exclude}")
        else:
            print(f"{seg_name} 无需排除 ICA 成分，保留所有组件")

        # 应用到 EEG
        # ICLabel要求：ICA拟合和应用的数据都应该使用相同的参考（平均参考）
        # 注意：在步骤4中已经对整个raw应用了平均参考，所以raw_eeg应该已经应用了平均参考
        # 如果使用ICLabel，我们需要确保raw_eeg也应用了平均参考（与raw_for_ica保持一致）
        # 但实际上，由于raw_eeg来自raw_seg，而raw_seg来自self.raw（已在步骤4应用平均参考），
        # 所以raw_eeg应该已经应用了平均参考
        # set_eeg_reference('average') 是幂等的，如果已经应用了平均参考，再次应用不会改变数据
        if use_iclabel:
            # 使用ICLabel时，为了保险起见，确保raw_eeg也应用了平均参考
            # 这样可以确保ICA拟合和应用的数据使用相同的参考系统
            # 由于set_eeg_reference是幂等的，即使已经应用了平均参考，再次应用也不会改变数据
            raw_eeg_for_apply = raw_eeg.copy().set_eeg_reference('average', projection=False, verbose=False)
        else:
            # 不使用ICLabel时，直接使用raw_eeg（应该已经应用了平均参考）
            raw_eeg_for_apply = raw_eeg.copy()
        
        # 注意：如果exclude为空，ica.apply()理论上不应该改变信号
        # 但由于数值精度和ICA重构过程，可能会有微小差异
        # 为了保持一致，我们统一使用ica.apply()方法
        if exclude:
            # 有需要排除的成分，应用ICA
            raw_eeg_clean = ica.apply(raw_eeg_for_apply, exclude=exclude, verbose=False)
            print(f"{seg_name} 已应用ICA，排除了 {len(exclude)} 个成分")
        else:
            # 如果没有需要排除的成分，直接返回原始数据（避免不必要的重构）
            print(f"{seg_name} 无需要排除的成分，保留原始数据")
            raw_eeg_clean = raw_eeg_for_apply.copy()

        # 是否保留AUX（ECG/EOG）通道
        # 如果只需要EEG通道，可以设置keep_aux_channels=False来跳过这部分
        if keep_aux_channels:
            # 把 AUX（ECG/EOG）加回
            # 作用：ICA处理时只对EEG通道进行处理，AUX通道被排除在外
            # 处理完后把AUX通道加回去，保持数据的完整性
            # 如果只需要EEG通道，可以设置keep_aux_channels=False来跳过这部分
            aux_present = [ch for ch in self.aux_channels if self._first_matching(ch, raw_seg.ch_names)]
            if aux_present:
                aux_raw = raw_seg.copy().pick_channels(aux_present)
                raw_seg_clean = raw_eeg_clean.copy().add_channels([aux_raw])
            else:
                raw_seg_clean = raw_eeg_clean
        else:
            # 只返回EEG通道，不包含AUX通道
            raw_seg_clean = raw_eeg_clean

        # 如果show_plot为True，显示处理后的数据
        if show_plot:
            # 如果show_plot是字符串且为'comparison'，显示对比；否则只显示处理后
            if isinstance(show_plot, str) and show_plot.lower() == 'comparison':
                # 显示对比（处理前后的数据）
                self._show_data_comparison(
                    raw_before=raw_for_ica,  # ICA处理前的原始数据
                    raw_after=raw_seg_clean,  # ICA处理后的数据
                    title_before=f"{seg_name} 段 - ICA处理前",
                    title_after=f"{seg_name} 段 - ICA处理后",
                    show_comparison=True
                )
            else:
                # 只显示处理后的数据
                self._show_data_plot(raw_seg_clean, title=f"{seg_name} 段 - ICA处理后的数据", show_plot=True)

        return raw_seg_clean


    def apply_ica(self, show_plot=False, eog_threshold=3.0, ecg_threshold='auto', 
                  emg_threshold=0.7, max_exclude_ratio=0.2, use_iclabel=False,
                  keep_aux_channels=False):
        """
        C.11: ICA伪迹去除
        对睁眼和闭眼段应用ICA处理。
        - ICA适合眼动/心电等可分离成分
        - 在连续数据上做更稳
        - ASR → ICA（先ASR后ICA）
        
        参数:
        ----------
        show_plot : bool or str, default=False
            是否显示处理后的数据。
            - False: 不显示
            - True: 只显示处理后的数据
            - 'comparison': 显示处理前后的对比（先显示处理前，再显示处理后）
        eog_threshold : float, default=3.0
            眼动伪迹检测阈值（标准差倍数），越高越保守
        ecg_threshold : float or str, default='auto'
            心跳伪迹检测阈值，'auto'表示自动阈值
        emg_threshold : float, default=0.7
            肌电伪迹检测阈值（0-1之间），越高越保守。原值为0.5，提高到0.7以减少过度排除
        max_exclude_ratio : float, default=0.3
            最大排除组件比例（0-1之间），避免一次性排除太多组件。例如0.3表示最多排除30%
        use_iclabel : bool, default=False
            是否使用ICLabel方法（需要安装mne-icalabel）。ICLabel是更智能的自动标注方法
        keep_aux_channels : bool, default=False
            是否保留AUX通道（ECG/EOG）。如果为False，只返回EEG通道。
            注意：ICA处理时只对EEG通道进行处理，AUX通道不会被ICA处理。
        """
        print("\n" + "="*60)
        print("C.11: ICA伪迹清理")
        print("="*60)

        self.raw_open_clean = self._ica_preprocess_segment(
            self.raw_open, "睁眼", show_plot=show_plot,
            eog_threshold=eog_threshold, ecg_threshold=ecg_threshold,
            emg_threshold=emg_threshold, max_exclude_ratio=max_exclude_ratio,
            use_iclabel=use_iclabel, keep_aux_channels=keep_aux_channels
        )
        self.raw_closed_clean = self._ica_preprocess_segment(
            self.raw_closed, "闭眼", show_plot=show_plot,
            eog_threshold=eog_threshold, ecg_threshold=ecg_threshold,
            emg_threshold=emg_threshold, max_exclude_ratio=max_exclude_ratio,
            use_iclabel=use_iclabel, keep_aux_channels=keep_aux_channels
        )


    # ========== D.13: 最终重参考 ==========
    def apply_final_rereference(self, ref_type='average', show_plot=False):
        """
        D.13: 最终重参考
        - 在坏道插值之后进行最终重参考
        - 对分段数据（睁眼段和闭眼段）分别进行重参考
        - 推荐使用平均参考（CAR）或其他参考方式
        
        参数:
        ----------
        ref_type : str, default='average'
            参考类型：'average'（平均参考CAR）或其他MNE支持的参考类型
        show_plot : bool, default=False
            是否在最后显示处理后的数据。如果为True，将使用MNE的plot方法显示数据。
        """
        print("\n" + "="*60)
        print("D.13: 最终重参考")
        print("="*60)
        print("对分段数据进行最终重参考...")
        
        if ref_type == 'average':
            # 对睁眼段重参考
            self.raw_open_clean.set_eeg_reference('average', projection=True, ch_type='eeg')
            self.raw_open_clean.apply_proj()
            print("  ✓ 睁眼段平均参考（CAR）完成")
            
            # 对闭眼段重参考
            self.raw_closed_clean.set_eeg_reference('average', projection=True, ch_type='eeg')
            self.raw_closed_clean.apply_proj()
            print("  ✓ 闭眼段平均参考（CAR）完成")
        else:
            # 对睁眼段重参考
            self.raw_open_clean.set_eeg_reference(ref_type)
            print(f"  ✓ 睁眼段{ref_type}参考完成")
            
            # 对闭眼段重参考
            self.raw_closed_clean.set_eeg_reference(ref_type)
            print(f"  ✓ 闭眼段{ref_type}参考完成")
        
        print("✓ D.13完成: 最终重参考")
        
        # 如果show_plot为True，显示处理后的数据
        if show_plot:
            self._show_data_plot(self.raw_open_clean, title="睁眼段 - 最终重参考后的数据", show_plot=True)
            self._show_data_plot(self.raw_closed_clean, title="闭眼段 - 最终重参考后的数据", show_plot=True)


    def save_results(self):
        """
        D.14: 保存
        - 保存清理后的数据
        """
        print("\n" + "="*60)
        print("D.14: 保存")
        print("="*60)
        
        subject_dir = Path(self.save_dir) / self.subject
        os.makedirs(subject_dir, exist_ok=True)
        print(f"\n保存路径: {subject_dir}")

        open_eyes_data = self.raw_open_clean.get_data(picks='eeg')
        closed_eyes_data = self.raw_closed_clean.get_data(picks='eeg')
        
        # 获取通道数
        n_channels = len(self.eeg_channels)
        
        # 检查并转置数据形状（确保最后一个维度等于通道数）
        if open_eyes_data.shape[-1] != n_channels:
            print(f"睁眼数据形状: {open_eyes_data.shape}，最后一个维度({open_eyes_data.shape[-1]})不等于通道数({n_channels})，进行转置...")
            # 转置最后两个维度
            open_eyes_data = np.swapaxes(open_eyes_data, -2, -1)
            print(f"转置后形状: {open_eyes_data.shape}")
        
        if closed_eyes_data.shape[-1] != n_channels:
            print(f"闭眼数据形状: {closed_eyes_data.shape}，最后一个维度({closed_eyes_data.shape[-1]})不等于通道数({n_channels})，进行转置...")
            # 转置最后两个维度
            closed_eyes_data = np.swapaxes(closed_eyes_data, -2, -1)
            print(f"转置后形状: {closed_eyes_data.shape}")

        np.save(subject_dir / 'open_eyes_clean_data.npy', open_eyes_data)
        np.save(subject_dir / 'closed_eyes_clean_data.npy', closed_eyes_data)

        print("\n--- 处理完成 ---")
        print("清理后数据: open_eyes_clean_data.npy, closed_eyes_clean_data.npy")
        print("✓ D.14完成: 保存")


    def _show_data_plot(self, raw_data, title="数据处理后的结果", show_plot=False):
        """
        封装的数据显示函数，用于在各个处理步骤后显示数据。
        
        参数:
        ----------
        raw_data : mne.io.Raw
            要显示的Raw数据对象
        title : str, default="数据处理后的结果"
            显示窗口的标题
        show_plot : bool, default=False
            是否显示数据。如果为True，将使用MNE的plot方法显示数据。
        """
        if not show_plot:
            return
        
        print(f"\n显示数据: {title}...")
        try:
            import matplotlib
            import matplotlib.pyplot as plt
            
            # 尝试设置交互式后端（如果还没有设置）
            current_backend = matplotlib.get_backend()
            interactive_backends = ['Qt5Agg', 'Qt4Agg', 'TkAgg', 'GTKAgg']
            
            if current_backend not in interactive_backends:
                try:
                    matplotlib.use('Qt5Agg', force=True)
                    # 重新导入pyplot以应用新后端
                    import importlib
                    importlib.reload(plt)
                    print(f"✓ 已切换到交互式后端: Qt5Agg")
                except Exception as e1:
                    try:
                        matplotlib.use('TkAgg', force=True)
                        import importlib
                        importlib.reload(plt)
                        print(f"✓ 已切换到交互式后端: TkAgg")
                    except Exception as e2:
                        print(f"⚠ 警告：无法切换到交互式后端")
                        print(f"  当前后端: {current_backend}")
                        print(f"  Qt5错误: {e1}")
                        print(f"  Tk错误: {e2}")
                        print("  提示：请在notebook开头运行 %matplotlib qt 或 %matplotlib tk")
            else:
                print(f"✓ 当前使用交互式后端: {current_backend}")
            
            # 获取数据长度用于设置合理的初始显示窗口
            data_duration = raw_data.times[-1] - raw_data.times[0]
            
            # 显示交互式窗口
            # 使用block=True确保窗口可以正常响应，但会阻塞直到窗口关闭
            # 这是MNE推荐的方式，可以确保窗口完全响应
            print("正在打开交互式数据窗口...")
            print("提示：窗口打开后，您可以：")
            print("  - 使用鼠标滚轮或窗口底部的滚动条查看不同时间段的数据")
            print("  - 使用键盘方向键（←→）可以前后移动时间窗口")
            print("  - 使用 +/- 键可以缩放时间轴")
            print("  - 点击通道名称可以标记/取消标记通道")
            print("  - 查看完毕后，关闭窗口以继续程序执行")
            print("")
            
            fig = raw_data.plot(
                title=title, 
                scalings='auto', 
                show=True,  # 显示窗口
                block=True,  # 阻塞模式，确保窗口完全响应，窗口关闭后继续执行
                duration=min(10.0, data_duration),  # 初始显示10秒或全部数据（取较小值）
                n_channels=min(20, len(raw_data.ch_names))  # 显示最多20个通道
            )
            print("✓ 数据窗口已关闭，继续执行...")
            
        except Exception as e:
            print(f"❌ 显示数据时出错: {e}")
            print("\n如果窗口无法正常显示或响应，请尝试以下方法：")
            print("  1. 在notebook开头运行: %matplotlib qt")
            print("  2. 或者运行: %matplotlib tk") 
            print("  3. 如果仍然无法交互，可能需要安装PyQt5: pip install PyQt5")
            print("  4. 然后重新运行此函数")
            import traceback
            traceback.print_exc()

    def _show_data_comparison(self, raw_before, raw_after, title_before="处理前", 
                              title_after="处理后", show_comparison=False):
        """
        对比显示处理前后的数据。
        
        参数:
        ----------
        raw_before : mne.io.Raw
            ICA处理前的Raw数据对象
        raw_after : mne.io.Raw
            ICA处理后的Raw数据对象
        title_before : str, default="处理前"
            处理前数据的显示窗口标题
        title_after : str, default="处理后"
            处理后数据的显示窗口标题
        show_comparison : bool, default=False
            是否显示对比。如果为True，将依次显示处理前后的数据窗口。
        """
        if not show_comparison:
            return
        
        try:
            import matplotlib
            import matplotlib.pyplot as plt
            
            # 尝试设置交互式后端（如果还没有设置）
            current_backend = matplotlib.get_backend()
            interactive_backends = ['Qt5Agg', 'Qt4Agg', 'TkAgg', 'GTKAgg']
            
            if current_backend not in interactive_backends:
                try:
                    matplotlib.use('Qt5Agg', force=True)
                    import importlib
                    importlib.reload(plt)
                    print(f"✓ 已切换到交互式后端: Qt5Agg")
                except Exception as e1:
                    try:
                        matplotlib.use('TkAgg', force=True)
                        import importlib
                        importlib.reload(plt)
                        print(f"✓ 已切换到交互式后端: TkAgg")
                    except Exception as e2:
                        print(f"⚠ 警告：无法切换到交互式后端")
                        print("  提示：请在notebook开头运行 %matplotlib qt 或 %matplotlib tk")
            
            # 获取数据长度用于设置合理的初始显示窗口
            data_duration_before = raw_before.times[-1] - raw_before.times[0]
            data_duration_after = raw_after.times[-1] - raw_after.times[0]
            data_duration = min(data_duration_before, data_duration_after)
            
            print(f"\n{'='*60}")
            print(f"显示数据对比: {title_before} vs {title_after}")
            print(f"{'='*60}")
            print("\n提示：将依次打开两个窗口，可以对比查看处理前后的数据")
            print("  - 第一个窗口：处理前的数据")
            print("  - 第二个窗口：处理后的数据")
            print("  - 使用鼠标滚轮或窗口底部的滚动条查看不同时间段的数据")
            print("  - 使用键盘方向键（←→）可以前后移动时间窗口")
            print("  - 使用 +/- 键可以缩放时间轴")
            print("  - 查看完毕后，关闭窗口以继续程序执行")
            print("")
            
            # 显示处理前的数据
            print(f"正在打开第一个窗口: {title_before}...")
            fig_before = raw_before.plot(
                title=title_before, 
                scalings='auto', 
                show=True,
                block=True,  # 阻塞模式，窗口关闭后继续
                duration=min(10.0, data_duration),
                n_channels=min(20, len(raw_before.ch_names))
            )
            print(f"✓ {title_before} 窗口已关闭")
            
            # 显示处理后的数据
            print(f"\n正在打开第二个窗口: {title_after}...")
            fig_after = raw_after.plot(
                title=title_after, 
                scalings='auto', 
                show=True,
                block=True,  # 阻塞模式，窗口关闭后继续
                duration=min(10.0, data_duration),
                n_channels=min(20, len(raw_after.ch_names))
            )
            print(f"✓ {title_after} 窗口已关闭")
            print(f"\n✓ 数据对比查看完成")
            
        except Exception as e:
            print(f"❌ 显示数据对比时出错: {e}")
            print("\n如果窗口无法正常显示或响应，请尝试以下方法：")
            print("  1. 在notebook开头运行: %matplotlib qt")
            print("  2. 或者运行: %matplotlib tk") 
            print("  3. 如果仍然无法交互，可能需要安装PyQt5: pip install PyQt5")
            print("  4. 然后重新运行此函数")
            import traceback
            traceback.print_exc()

            
    def run_preprocessing(self, data_file_path, subject, 
                         interpolate_bads=False, bad_channels=None, 
                         auto_detect_bads=True, show_plots=[False, False, False, False, False],
                         save_data=True, use_asr=True, use_ica=True, **kwargs):
        """
        执行完整的预处理流程（按照优化后的流程）。
        
        流程概述：
        A. 质控与设置：加载数据、通道设置、montage、临时重参考、标记坏道、查找睁/闭眼区段
        B. 信号整形：滤波、降采样
        C. 伪迹去除：裁剪睁/闭眼段、ASR、ICA
        D. 修补与定稿：坏道插值、最终重参考、分段和保存
        
        参数:
        ----------
        data_file_path : str
            数据文件路径（.bdf，已包含事件信息）
        subject : str
            被试名称
        interpolate_bads : bool, default=False
            是否执行坏导联插值（在D.9步骤，ASR/ICA之后）
        bad_channels : list of str, optional
            手动指定的坏导联列表（在A.5步骤标记，D.9步骤插值）
        auto_detect_bads : bool, default=True
            是否自动检测坏导联（在A.5步骤）
        show_plots : list of bool, default=[False, False, False, False, False, False]
            是否显示处理后的数据。如果为True，将在处理完每个段后显示数据。
            索引对应：[临时重参考, 滤波, 降采样, ASR, ICA, 最终重参考]
        save_data : bool, default=True
            是否保存数据，只有为True时，数据才会保存到save_dir中
        use_asr : bool, default=True
            是否使用ASR进行大伪迹清理。如果为False，将跳过ASR处理步骤。
        use_ica : bool, default=True
            是否使用ICA进行伪迹去除。如果为False，将跳过ICA处理步骤。
        **kwargs : dict
            其他处理参数，包括：
            - ASR相关参数（仅在use_asr=True时有效）：
              - asr_cutoff: float, ASR截止阈值（默认20）
              - asr_show_plot: bool or str, 是否显示ASR处理后的数据
            - ICA相关参数（仅在use_ica=True时有效）：
              - show_plot: bool or str, 是否显示ICA处理后的数据
              - eog_threshold: float, 眼动伪迹检测阈值
              - ecg_threshold: float or str, 心跳伪迹检测阈值
              - emg_threshold: float, 肌电伪迹检测阈值
              - max_exclude_ratio: float, 最大排除组件比例
              - use_iclabel: bool, 是否使用ICLabel方法
              - keep_aux_channels: bool, 是否保留AUX通道
        """
        self.data_file_path = data_file_path
        self.subject = subject
        
        # ========== A. 质控与设置 ==========
        # A.1: 数据加载（保留原始参考信息）
        self.load_data_and_check()
        
        # A.2: 通道设置
        self.select_and_setup_channels()
        
        # A.3: 设置 montage（电极位置非常重要：后续插值要用）
        self.setup_montage()
        
        # A.4: 临时重参考用于目视检查（可选，不保存、不进入正式分支）
        temp_ref_show_plot = kwargs.get('temp_ref_show_plot', show_plots[0] if len(show_plots) > 0 else False)
        self.apply_temp_rereference(ref_type='average', show_plot=temp_ref_show_plot)
        
        # A.5: 人工标记坏道（只标记，不插值）
        # 注意：这里只标记，插值将在D.9进行
        if interpolate_bads and len(bad_channels) > 0:
            self.mark_bad_channels(
                bad_channels=bad_channels, auto_detect=auto_detect_bads
            )
        
        # A.6: 查找睁/闭眼区段
        self.define_segments()
        
        # ========== B. 信号整形 ==========
        # B.6: 滤波（高通 0.5/1 Hz，低通 45 Hz，notch 50/60 Hz）
        self.apply_filtering(show_plot=show_plots[1] if len(show_plots) > 1 else False)
        
        # B.7: 降采样（低通后做，避免混叠）
        self.apply_downsampling(show_plot=show_plots[2] if len(show_plots) > 2 else False)
        
        # ========== C. 伪迹去除（连续数据上做更稳）==========
        if save_data:
            # C.9: 裁剪睁/闭眼段（用于ASR/ICA处理）
            self.crop_segments()
            
            # C.10: ASR伪迹清理（可选，在ICA之前）
            if use_asr:
                asr_cutoff = kwargs.get('asr_cutoff', 20)
                asr_show_plot = kwargs.get('asr_show_plot', show_plots[3] if len(show_plots) > 3 else False)
                self.apply_asr(cutoff=asr_cutoff, show_plot=asr_show_plot)
            else:
                print("\n" + "="*60)
                print("C.10: ASR伪迹清理（跳过）")
                print("="*60)
                print("跳过ASR处理，直接使用原始分段数据")
                print("✓ C.10完成: 跳过ASR处理")
            
            # C.11: ICA伪迹去除（可选）
            if use_ica:
                # 提取ICA相关参数（排除ASR相关参数）
                ica_kwargs = {k: v for k, v in kwargs.items() 
                                if not k.startswith('asr_') and k != 'channels_to_drop' and k != 'temp_ref_show_plot'}
                # 如果ica_kwargs中没有指定show_plot，则使用show_plots[4]
                if 'show_plot' not in ica_kwargs:
                    ica_kwargs['show_plot'] = show_plots[4] if len(show_plots) > 4 else False
                self.apply_ica(**ica_kwargs)
            else:
                # 不使用ICA，直接使用ASR处理后的分段数据（如果使用了ASR）
                print("\n" + "="*60)
                print("C.11: ICA伪迹去除（跳过）")
                print("="*60)
                print("跳过ICA处理，直接使用分段数据")
                if use_asr:
                    # 如果使用了ASR，使用ASR处理后的数据
                    self.raw_open_clean = self.raw_open.copy()
                    self.raw_closed_clean = self.raw_closed.copy()
                else:
                    # 如果都没使用，使用原始分段数据
                    self.raw_open_clean = self.raw_open.copy()
                    self.raw_closed_clean = self.raw_closed.copy()
                print("✓ C.11完成: 跳过ICA处理")
            
            # ========== D. 修补与定稿 ==========
            # D.12: 坏道插值（在ASR/ICA之后）
            if interpolate_bads and len(self.bad_channels) > 0:
                # 对分段数据分别进行插值
                print("\n" + "="*60)
                print("D.12: 坏道插值（在ASR/ICA之后）")
                print("="*60)
                print("对分段数据进行插值...")
                
                # 对睁眼段插值
                if len(self.raw_open_clean.info.get('bads', [])) == 0:
                    self.raw_open_clean.info['bads'] = self.bad_channels.copy()
                try:
                    self.raw_open_clean.interpolate_bads(reset_bads=True, mode='accurate', verbose=False)
                    print(f"  ✓ 睁眼段插值完成")
                except Exception as e:
                    print(f"  ⚠ 睁眼段插值失败: {e}")
                
                # 对闭眼段插值
                if len(self.raw_closed_clean.info.get('bads', [])) == 0:
                    self.raw_closed_clean.info['bads'] = self.bad_channels.copy()
                try:
                    self.raw_closed_clean.interpolate_bads(reset_bads=True, mode='accurate', verbose=False)
                    print(f"  ✓ 闭眼段插值完成")
                except Exception as e:
                    print(f"  ⚠ 闭眼段插值失败: {e}")
                
                # 清除坏道标记
                self.raw_open_clean.info['bads'] = []
                self.raw_closed_clean.info['bads'] = []
                self.bad_channels = []
                print("✓ D.12完成: 坏道插值")
            else:
                print("\n" + "="*60)
                print("D.12: 坏道插值（跳过）")
                print("="*60)
                print("未标记坏道或interpolate_bads=False，跳过插值")
                print("✓ D.12完成: 跳过坏道插值")
            
            # D.13: 最终重参考（平均参考CAR）
            final_ref_show_plot = kwargs.get('final_ref_show_plot', show_plots[5] if len(show_plots) > 5 else False)
            self.apply_final_rereference(ref_type='average', show_plot=final_ref_show_plot)
            
            # D.14: 保存
            self.save_results()

In [8]:
class CAEEG_BatchProcessor:
    """
    用于批量处理 E:\YandaDataset\Assess-01 文件夹下所有被试的数据。
    对每个被试：
        - 自动找到采集时间最早的 session（形如 20250916102803_Assess-01_xxx）
        - 定位以受试者姓名命名的 bdf 文件（如 subject_name.bdf，已包含事件信息）
        - 调用 USTB_MCI_preprocessing 类执行完整预处理流程
    """

    def __init__(self, l_freq=0.5, h_freq=45.0, sfreq=128, keep_channels=None,
                 root_dir=None, save_dir=None, skip_if_exists=True, resting_state=None,
                 experiments_number='Assess-01', log_file_name='preprocessing_log.txt'):
        """
        初始化批处理器。
        参数
        ----------
        l_freq : float
            低频滤波阈值
        h_freq : float
            高频滤波阈值
        sfreq : float
            降采样后的采样率
        keep_channels : list
            需要保留的通道列表
        root_dir : str
            根目录路径（包含所有被试文件夹）
        save_dir : str
            保存预处理结果的目录
        skip_if_exists : bool
            若目标被试下已有处理结果文件，则跳过
        resting_state : str
            预处理结果保存的文件名后缀（可选）
        experiments_number : str, default='Assess-01'
            实验编号，用于匹配session文件夹名称（如 'Assess-01' 或 'Assess-02'）
        log_file_name : str, default='preprocessing_log.txt'
            日志文件名，保存在save_dir目录下
        """
        self.root_dir = Path(root_dir)
        self.save_dir = Path(save_dir)
        self.skip_if_exists = skip_if_exists
        self.experiments_number = experiments_number
        self.log_file_name = log_file_name
        self.time_pattern = re.compile(rf"(\d{{14}})_{re.escape(experiments_number)}_([A-Za-z0-9_]+)")
        self.preproc = CAEEG_preprocessing(l_freq, h_freq, sfreq, keep_channels, save_dir=self.save_dir)
        self.resting_state = resting_state

    # ------------------------------------
    # 工具函数：查找最早 session
    # ------------------------------------
    def find_earliest_session(self, subject_dir: Path):
        """返回该被试最早时间的 session 文件夹 Path"""
        subfolders = [f for f in subject_dir.iterdir() if f.is_dir()]
        sessions = []
        for sf in subfolders:
            match = self.time_pattern.match(sf.name)
            if match:
                timestamp = match.group(1)
                try:
                    dt = datetime.strptime(timestamp, "%Y%m%d%H%M%S")
                    sessions.append((dt, sf))
                except ValueError:
                    continue
        if not sessions:
            return None
        sessions.sort(key=lambda x: x[0])
        return sessions[0][1]

    def find_latest_session(self, subject_dir: Path):
        """返回该被试最晚时间的 session 文件夹 Path"""
        subfolders = [f for f in subject_dir.iterdir() if f.is_dir()]
        sessions = []
        for sf in subfolders:
            match = self.time_pattern.match(sf.name)
            if match:
                timestamp = match.group(1)
                try:
                    dt = datetime.strptime(timestamp, "%Y%m%d%H%M%S")
                    sessions.append((dt, sf))
                except ValueError:
                    continue
        if not sessions:
            return None
        sessions.sort(key=lambda x: x[0], reverse=True)
        return sessions[0][1]

    # ------------------------------------
    # 工具函数：找到 subject_name.bdf
    # ------------------------------------
    def find_subject_bdf(self, session_folder: Path, subject_name: str):
        """在 session 目录下递归查找以受试者姓名命名的bdf文件"""
        bdf_filename = f"{subject_name}.bdf"
        for root, dirs, files in os.walk(session_folder):
            if bdf_filename in files:
                bdf_path = Path(root) / bdf_filename
                return bdf_path
        return None

    # ------------------------------------
    # 检查是否已有输出
    # ------------------------------------
    def already_processed(self, subject_name: str) -> bool:
        """判断该被试是否已有输出文件"""
        result_files = [
            f"open_eyes_clean_data.npy",
            f"closed_eyes_clean_data.npy",
        ]
        # 查找被试输出文件夹（检查 save_dir 而不是 root_dir）
        for rf in result_files:
            if (self.save_dir / subject_name / rf).exists():
                return True
        return False

    # ------------------------------------
    # 处理单个被试
    # ------------------------------------
    def process_subject(self, subject_dir: Path, review_data = False):
        subj_name = subject_dir.name
        print(f"\n=== 开始处理被试: {subj_name} ===")

        # 是否跳过
        if self.skip_if_exists and self.already_processed(subj_name):
            print(f"已检测到输出文件，跳过 {subj_name}")
            return

        if self.resting_state == 'befor':
            # 找最早 session
            session = self.find_earliest_session(subject_dir)
            if session is None:
                print(f"{subj_name} 未找到有效 session，跳过。")
                return
        elif self.resting_state == 'after':
            # 找最晚 session
            session = self.find_latest_session(subject_dir)
            if session is None:
                print(f"{subj_name} 未找到有效 session，跳过。")
                return

        # 找 bdf 文件（以受试者姓名命名）
        bdf_file = self.find_subject_bdf(session, subj_name)
        if bdf_file is None:
            print(f"{subj_name} 的 {session} 中缺少 {subj_name}.bdf 文件，跳过。")
            return
        print(f"采集时间: {session.name}")
        print(f"BDF文件: {bdf_file}")

        # 调用预处理流程
        try:
            self.preproc.run_preprocessing(
                str(bdf_file), subj_name,
                interpolate_bads=True,
                bad_channels=['Cz'],  # 'Pz', 'Oz', 'Fz'
                auto_detect_bads=False,
                show_plots=[review_data, False, False, False, False, False], # [临时重参考, 滤波, 降采样, ASR, ICA，最终重参考]
                # 是否保存数据，只有为True时，数据才会保存到save_dir中
                save_data=not review_data,
                use_asr=True,  # 是否使用ASR处理，可以设置为False跳过ASR
                use_ica=True,  # 是否使用ICA处理，可以设置为False跳过ICA
                # 以下为ASR相关参数（可选）
                asr_cutoff=20,          # ASR截止阈值
                # 以下为ICA相关参数（可选）
                emg_threshold=0.7,        # 肌电伪迹检测阈值
                max_exclude_ratio=0.2,   # 最大排除组件比例
                use_iclabel=True        # 是否使用ICLabel方法
            )
            print(f"{subj_name} 处理完成。")
        except Exception as e:
            print(f"处理 {subj_name} 时出错: {e}")
            import traceback
            traceback.print_exc()

    # ------------------------------------
    # 批量执行
    # ------------------------------------
    def run_all(self):
        # 初始化日志文件（所有受试者共用一个日志文件）
        log_file_path = Path(self.save_dir) / self.log_file_name
        self.preproc._init_log_file(log_file_path=log_file_path)
        
        try:
            subjects = [d for d in self.root_dir.iterdir() if d.is_dir()]
            print(f"在 {self.root_dir} 下发现 {len(subjects)} 个被试。")
            print(f"日志文件: {log_file_path}\n")

            for subj_dir in subjects:
                # if subj_dir.name == 'yanruhan':
                #     # 初始化日志文件（所有受试者共用一个日志文件）
                #     self.preproc._init_log_file(subject=subj_dir.name)
                self.process_subject(subj_dir, review_data = False) # True 查看数据， False 不查看数据
            print("\n所有被试处理完成。")
        finally:
            # 确保日志文件被关闭
            self.preproc._close_log_file()

In [9]:
if __name__ == "__main__":

    l_freq, h_freq, sfreq = 0.5, 45.0, 128
    experiments_number = 'Assess-01' # 'Assess-01' or 'Assess-02'
    resting_state = "after" # "befor" or "after"
    root_dir = os.path.join(r"F:\CAEEG\Raw data\Multimodal data", experiments_number)
    save_dir = os.path.join(r"E:\CAEEG " + str(sfreq) + r"Hz", experiments_number, "resting state " + resting_state)
    log_file_name = "preprocessing_log.txt"  # 日志文件名，可根据需要自定义

    keep_channels = [
            'Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8',
            'Cz', 'C3', 'C4', 'T7', 'T8', 'Pz', 'P3', 'P4', 'P7', 'P8',
            'Oz', 'O1', 'O2', 'ECG', 'HEOR', 'HEOL'
        ]

    processor = CAEEG_BatchProcessor(
        l_freq=l_freq, h_freq=h_freq, sfreq=sfreq, keep_channels=keep_channels,
        root_dir=root_dir,
        save_dir=save_dir,
        skip_if_exists=True,  # 若检测到已有输出则自动跳过
        resting_state=resting_state,
        experiments_number=experiments_number,  # 传入实验编号
        log_file_name=log_file_name  # 传入日志文件名
    )
    processor.run_all()

In [10]:
if __name__ == "__main__":

    l_freq, h_freq, sfreq = 0.5, 45.0, 128
    experiments_number = 'Assess-02' # 'Assess-01' or 'Assess-02'
    resting_state = "after" # "befor" or "after"
    root_dir = os.path.join(r"F:\CAEEG\Raw data\Multimodal data", experiments_number)
    save_dir = os.path.join(r"E:\CAEEG " + str(sfreq) + r"Hz", experiments_number, "resting state " + resting_state)
    log_file_name = "preprocessing_log.txt"  # 日志文件名，可根据需要自定义

    keep_channels = [
            'Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8',
            'Cz', 'C3', 'C4', 'T7', 'T8', 'Pz', 'P3', 'P4', 'P7', 'P8',
            'Oz', 'O1', 'O2', 'ECG', 'HEOR', 'HEOL'
        ]

    processor = CAEEG_BatchProcessor(
        l_freq=l_freq, h_freq=h_freq, sfreq=sfreq, keep_channels=keep_channels,
        root_dir=root_dir,
        save_dir=save_dir,
        skip_if_exists=True,  # 若检测到已有输出则自动跳过
        resting_state=resting_state,
        experiments_number=experiments_number,  # 传入实验编号
        log_file_name=log_file_name  # 传入日志文件名
    )
    processor.run_all()

在 F:\CAEEG\Raw data\Multimodal data\Assess-02 下发现 65 个被试。
日志文件已创建: E:\CAEEG 128Hz\Assess-02\resting state befor\yanruhan_preprocessing_log.txt

=== 开始处理被试: yanruhan ===
采集时间: 20251202153032_Assess-02_yanruhan
BDF文件: F:\CAEEG\Raw data\Multimodal data\Assess-02\yanruhan\20251202153032_Assess-02_yanruhan\20251202153032_Assess-02_yanruhan\yanruhan\yanruhan.bdf

A.1: 数据加载（保留原始参考信息）
正在加载数据文件: F:\CAEEG\Raw data\Multimodal data\Assess-02\yanruhan\20251202153032_Assess-02_yanruhan\20251202153032_Assess-02_yanruhan\yanruhan\yanruhan.bdf
✓ 数据文件加载完成
  - 采样率: 1000.0 Hz
  - 通道数量: 73
  - 数据时长: 421.00 秒
  - 数据形状: (73, 421000)
  - 原始参考信息: 未应用自定义参考
  ✓ 保留原始参考信息，暂不进行重参考
✓ 从BDF文件中读取到 6 个注解
  - 事件数量: 6
  - 事件类型: [np.str_('10'), np.str_('11'), np.str_('12'), np.str_('13'), np.str_('14'), np.str_('15')]
✓ A.1完成: 数据加载（保留原始参考信息）

A.2: 通道设置
原始通道列表 (73 个):
  ['Fpz', 'Fp1', 'Fp2', 'AF3', 'AF4', 'AF7', 'AF8', 'Fz', 'F1', 'F2']...

1. 筛选通道...
NOTE: pick_channels() is a legacy function. New code should use inst.pick